# 04 – Start the web app

Starts the prototype's web app (`prototype/app.py`) inside this notebook, on the JupyterHub GPU.

Run all cells from the top. The last cell starts the app and prints a link (`https://....gradio.live`);
open it in a new browser tab.
**The app runs as long as the last cell runs** (it shows `[*]`). To stop the app, press the **Stop button (■)**
in the toolbar.

The app is opened through a temporary `gradio.live` link: the JupyterHub computer's own address (127.0.0.1) cannot be
opened from another computer, so Gradio sends the page through its own server to the browser. Anyone who has this link can use the app while it runs, so share it only with people who
should test it and stop the app afterwards.

## Setup (same cells as in the other notebooks)

In [1]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"   # hides TensorFlow's internal C++ log lines (GPU timer notes etc.); must be set before TensorFlow is imported

import sys
import subprocess
import importlib
import warnings
from pathlib import Path

# google-api-core warns about a grpcio version in 2027 - not relevant here
warnings.filterwarnings("ignore", category=FutureWarning, module="google.api_core")


def pip_install(*arguments):
    # quiet install; pip's notes about other packages on the JupyterHub are only shown if the install fails
    result = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--disable-pip-version-check", *arguments],
                            capture_output=True, text=True)
    if result.returncode != 0:
        print(result.stdout[-3000:], result.stderr[-3000:])
        raise RuntimeError("pip install failed: " + " ".join(arguments))


# the prototype packages (versions as in requirements.txt); TensorFlow, Keras, NumPy, pandas and matplotlib
# are already installed on the JupyterHub
pip_install("earthengine-api==1.7.49", "rasterio==1.4.4", "gradio==6.31.0")
importlib.invalidate_caches()          # lets Python find packages that were installed just now
for module in ["ee", "rasterio", "gradio"]:
    importlib.import_module(module)
print("packages ready")

packages ready


In [2]:
import tensorflow as tf

# take GPU memory only when it is needed, so this notebook also runs while another notebook holds some of it
# (must run before the first model is loaded; after a kernel restart this cell has to come first)
try:
    for gpu in tf.config.list_physical_devices("GPU"):
        tf.config.experimental.set_memory_growth(gpu, True)
    print("GPUs:", tf.config.list_physical_devices("GPU"))
except RuntimeError:
    print("the GPU was already in use in this kernel - restart the kernel and run all cells from the top")

GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [3]:
from importlib.metadata import version as package_version, PackageNotFoundError

# keras-hub (SegFormer) is installed WITHOUT its dependencies: its metadata asks for tensorflow-text, which would
# replace the installed TensorFlow. keras-hub 0.24.0 works with Keras 3.8 - 3.14 (tested: TensorFlow 2.20, Keras 3.11.2).
KERAS_HUB = "0.24.0"
try:
    installed = package_version("keras-hub")
except PackageNotFoundError:
    installed = None
if installed != KERAS_HUB:
    print(f"installing keras-hub {KERAS_HUB} (found: {installed})")
    pip_install("--no-deps", "--force-reinstall", f"keras-hub=={KERAS_HUB}")
    if "keras_hub" in sys.modules:
        raise RuntimeError("keras-hub was replaced - restart the kernel and run all cells again")

# small packages keras-hub needs (they do not depend on TensorFlow)
missing = []
for package in ["kagglehub", "huggingface_hub", "regex", "rich"]:
    try:
        package_version(package.replace("_", "-"))
    except PackageNotFoundError:
        missing.append(package)
if missing:
    pip_install(*missing)

import keras_hub
print("keras-hub:", keras_hub.__version__)

installing keras-hub 0.24.0 (found: None)
keras-hub: 0.24.0


In [4]:
import ee

PROJECT_ID = "sentinel1-lulc-project-512618"

# Only asks for a new login when the saved token is missing. Clear this cell's output before sharing.
try:
    ee.Initialize(project=PROJECT_ID)
except Exception:
    ee.Authenticate(auth_mode="notebook")
    ee.Initialize(project=PROJECT_ID)
print("Earth Engine ready:", ee.Number(1).getInfo())


Successfully saved authorization token.
Earth Engine ready: 1


## Load the models

Both modes are loaded before the app starts, so the first request does not have to wait for it.

In [5]:
sys.path.insert(0, "prototype")
import app

app.MODEL_DIR = "models/round3"
app.PROJECT_ID = PROJECT_ID
print("standard ->", app.get_model("standard")[2]["model"])
try:
    print("ensemble ->", app.get_model("ensemble")[2]["model"])
except FileNotFoundError as error:
    # the ensemble needs the four best.keras files in models/round3/<model>/ (see README, section 6)
    print("ensemble mode not available:", error)

standard -> segformer_b3


ensemble -> ensemble of 4 models (segformer_b3, segformer_b2, deeplabv3plus_resnet50, unet_resnet50), 8 flips/rotations each


## Start the app

Open the `https://....gradio.live` link printed below in a new browser tab.
The cell keeps running while the app is open. Stop it with the Stop button (■) in the toolbar.

In [6]:
# quiet=True: Gradio does not print its own addresses (its 127.0.0.1 address only works on the JupyterHub computer)
demo = app.start(share=True, prevent_thread_lock=True, show_error=True, inline=False, quiet=True)
print("Open this link in a new browser tab:", demo.share_url or demo.local_url)
print("The app is running. Stop it with the Stop button (■) in the toolbar.")
demo.block_thread()       # keeps the app open until the Stop button is pressed
print("app stopped")

* Running on public URL: https://a2f7584dc46e723d30.gradio.live
Open this link in a new browser tab: https://a2f7584dc46e723d30.gradio.live
The app is running. Stop it with the Stop button (■) in the toolbar.
Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://a2f7584dc46e723d30.gradio.live
app stopped
